# Example 06c: preparing the real MAST-U machine for up-down symmetric calculations

This example starts from the standard **series-connected MAST-U active-coil
description used when symmetric_machine=True in example 06a**. In that file,
each named shaping circuit already contains its upper and lower filament bundles
in series. The word "symmetric" describes this electrical grouping; the source
geometry is only approximately up-down symmetric.

The notebook workflow has four distinct operations:

1. inspect the reflection parity of each series-connected active circuit;
2. infer and remove a common vertical offset of the complete machine;
3. identify and average reflected conductor pairs, limiter, and wall;
4. independently quantify the active/passive magnetic-fingerprint changes and
   inspect the supporting geometry differences.

The machine description is fetched directly from UDA for the selected MAST-U
shot and retained in memory: this notebook does not depend on, create, or load
machine-description files. The original dictionaries are then deep-copied and
are not modified.

**Note:** this notebook requires access to a UDA client, as in examples 06a and
06b.

## Imports and UDA source data

We request the series-connected active description returned by
`mastu_tools.get_machine_data`. This gives one current coordinate per series
circuit. Passing no `save_path` keeps every description in memory and avoids a
dependency on local machine files.

Series connection alone does not guarantee even-in-Z magnetic parity. The P6
circuit is the important special case: its lower bundle has the opposite
polarity, so P6 is odd and must be omitted from a purely even evolution.

In [ ]:
from copy import deepcopy
from html import escape
import os

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

from freegsnke import build_machine, equilibrium_update, mastu_tools
from freegsnke.up_down_symmetry import prepare_up_down_symmetric_machine

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

def display_records(records, columns):
    """Display selected dictionary fields without adding a pandas dependency."""
    def formatted(value):
        if isinstance(value, float):
            return f"{value:.6g}"
        return str(value)

    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    rows = "".join(
        "<tr>"
        + "".join(f"<td>{escape(formatted(record[column]))}</td>" for column in columns)
        + "</tr>"
        for record in records
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{rows}</tbody></table>"))

# Configure the UDA client in the same way as examples 06a and 06b.
os.environ["UDA_HOST"] = "uda2.hpc.l"
# os.environ["UDA_HOST"] = "data.mastu.ukaea.uk"
os.environ["UDA_PORT"] = "56565"
os.environ["UDA_META_PLUGINNAME"] = "MASTU_DB"
os.environ["UDA_METANEW_PLUGINNAME"] = "MASTU_DB"

shot = 53320
machine_data = mastu_tools.get_machine_data(
    shot=shot,
    split_passives=True,
)

active_series_raw = machine_data["active_coils_data"]
passive_raw = machine_data["passive_coils_data"]
limiter_raw = machine_data["limiter_data"]
wall_raw = machine_data["wall_data"]

print(f"Series-connected active circuits in source: {len(active_series_raw)}")
print(f"Passive structures:                         {len(passive_raw)}")
print(f"Limiter vertices:                           {len(limiter_raw)}")

## Step 1: safety check, then explicit preparation

There is no separate *pre-flight mode* in the API. The first call below uses
the normal `prepare_up_down_symmetric_machine()` function with its default
safety policy. On private copies of the inputs, it:

1. identifies upper/lower and internally symmetric elements;
2. fits the common source midplane when `z_midplane="auto"`;
3. checks whether each active circuit is electrically even or odd under
   reflection.

If any odd active circuit is found, the default call raises before returning a
prepared machine. This intentional failure is the safety check: it prevents a
source circuit from disappearing silently. It does not modify the source
description and it does not construct a machine for the solver.

The next call makes the modelling decision explicit with
`exclude_odd_active=True`. It reruns the complete pipeline, removes and records
the odd source circuits, averages retained geometry onto exact reflection
symmetry, and returns the `PreparedUpDownMachine` used below.


In [ ]:
# Demonstration only: run preparation with its strict default policy to
# show which circuits prevent strict even evolution. Once that modelling
# decision is known, normal user code can omit this try/except call.
# This call is expected to stop at the electrical-parity check because
# MAST-U P6 is odd. The input dictionaries are deep-copied internally.
try:
    prepare_up_down_symmetric_machine(
        active_series_raw,
        passive_raw,
        limiter_data=limiter_raw,
        wall_data=wall_raw,
        z_midplane="auto",
    )
except ValueError as error:
    print(f"Safety check stopped preparation: {error}")

# Functional call: explicitly accept exclusion of odd sources and build the
# recentered, exactly symmetric description used from this point onward.
prepared = prepare_up_down_symmetric_machine(
    active_series_raw,
    passive_raw,
    limiter_data=limiter_raw,
    wall_data=wall_raw,
    z_midplane="auto",
    exclude_odd_active=True,
)

# Preserve the identity of excluded sources and construct the retained
# unsymmetrised active basis needed for the magnetic fingerprint audit.
odd_series_circuits = prepared.excluded_odd_active_names
active_raw = {
    name: deepcopy(active_series_raw[name])
    for name in prepared.original_active_names
}

print(f"Electrically odd circuits excluded: {odd_series_circuits}")
print(f"Fitted source midplane: {1e3 * prepared.source_z_midplane:+.3f} mm")
print(f"Applied whole-machine shift: {1e3 * prepared.z_shift:+.3f} mm")
print(f"Pointwise midplane spread: {1e3 * prepared.midplane_fit_rms:.3f} mm")
print(f"Matched point pairs in fit: {prepared.midplane_fit_samples}")

## Step 2: decide from the magnetic fingerprints

These are the primary acceptance quantities. They compare the sampled Green
function of every retained source element with the corresponding recentered and
symmetrised element on the same grid. The reported percentage is
`100 * norm(G_sym - G_raw) / norm(G_raw)`, using the Frobenius norm.

The aggregate percentages describe the complete active and passive bases and
do not depend on choosing a particular current vector. The table also names the
single element with the largest relative field change. These magnetic changes,
rather than millimetre geometry differences alone, should decide whether the
symmetrised approximation is acceptable for the intended calculation.

In [ ]:
# Build two machines with identical retained current coordinates: one
# from the otherwise unmodified source and one from prepared geometry.
# Equal ordering is essential for element-by-element Green comparisons.
raw_audit_tokamak = build_machine.tokamak(
    active_coils_data=active_raw,
    passive_coils_data=passive_raw,
    limiter_data=limiter_raw,
    wall_data=wall_raw,
)
# Equilibrium supplies the common (R, Z) grid and sampled conductor Green
# functions. No Grad-Shafranov solve is needed for this basis comparison.
raw_audit_eq = equilibrium_update.Equilibrium(
    tokamak=raw_audit_tokamak,
    Rmin=0.06,
    Rmax=2.0,
    Zmin=-2.2,
    Zmax=2.2,
    nx=65,
    ny=65,
    psi=None,
)

symmetric_audit_tokamak = build_machine.tokamak(
    active_coils_data=prepared.active_coils_data,
    passive_coils_data=prepared.passive_coils_data,
    limiter_data=prepared.limiter_data,
    wall_data=prepared.wall_data,
)
symmetric_audit_eq = equilibrium_update.Equilibrium(
    tokamak=symmetric_audit_tokamak,
    Rmin=0.06,
    Rmax=2.0,
    Zmin=-2.2,
    Zmax=2.2,
    nx=65,
    ny=65,
    psi=None,
)

# _vgreen is the internal conductor-to-grid poloidal-flux basis. It stores
# active rows first and passive rows second, so split both
# arrays at the same retained-active count before comparing the bases.
n_active = len(prepared.original_active_names)
raw_active_greens = raw_audit_eq._vgreen[:n_active]
sym_active_greens = symmetric_audit_eq._vgreen[:n_active]
raw_passive_greens = raw_audit_eq._vgreen[n_active:]
sym_passive_greens = symmetric_audit_eq._vgreen[n_active:]


def percentage_change(original, updated):
    """Return 100 times the relative Frobenius-norm change."""
    return 100 * np.linalg.norm(updated - original) / np.linalg.norm(original)


# Aggregate norms answer whether the complete magnetic basis changed;
# per-element norms identify which conductor contributes most.
active_fingerprint_change_percent = percentage_change(
    raw_active_greens, sym_active_greens
)
passive_fingerprint_change_percent = percentage_change(
    raw_passive_greens, sym_passive_greens
)
active_element_changes = np.array(
    [percentage_change(raw, sym) for raw, sym in zip(raw_active_greens, sym_active_greens)]
)
passive_element_changes = np.array(
    [percentage_change(raw, sym) for raw, sym in zip(raw_passive_greens, sym_passive_greens)]
)
worst_active = int(np.argmax(active_element_changes))
worst_passive = int(np.argmax(passive_element_changes))

fingerprint_rows = [
    {
        "system": "active coils",
        "aggregate_change_percent": active_fingerprint_change_percent,
        "largest_element": prepared.original_active_names[worst_active],
        "largest_element_change_percent": active_element_changes[worst_active],
    },
    {
        "system": "passive structures",
        "aggregate_change_percent": passive_fingerprint_change_percent,
        "largest_element": prepared.passive_names[worst_passive],
        "largest_element_change_percent": passive_element_changes[worst_passive],
    },
]

print("MAGNETIC FINGERPRINT CHANGES")
print(f"Active Green basis:  {active_fingerprint_change_percent:.4f}%")
print(f"Passive Green basis: {passive_fingerprint_change_percent:.4f}%")
display_records(
    fingerprint_rows,
    [
        "system",
        "aggregate_change_percent",
        "largest_element",
        "largest_element_change_percent",
    ],
)

## Step 3: inspect the supporting geometry differences

The geometry distances explain where the magnetic changes came from, but are
secondary diagnostics. `geometry_discrepancies` covers separately named active
and passive pairs, filament bundles inside series circuits, self-symmetric
structures, the limiter, and the wall. Each value is the pre-averaging RMS
distance between an upper geometry and its reflected lower partner.

The table shows the largest changes first. After accepting the magnetic
fingerprint changes, the user can still impose an absolute geometric guard with
`check_geometry_tolerance()`. Passing `max_pair_mismatch` to the preparation
function performs the same check immediately.

In [ ]:
# These records were measured on the source geometry before pair averaging,
# even though prepared already contains the resulting symmetric copies.
geometry_rows = [
    vars(record) | {"reflected_rms_mm": 1e3 * record.reflected_rms}
    for record in prepared.largest_geometry_discrepancies()
]
largest = prepared.maximum_geometry_discrepancy

print(
    "Largest proposed geometric correction: "
    f"{1e3 * largest.reflected_rms:.3f} mm in {largest.component} "
    f"'{largest.upper_name}' / '{largest.lower_name}'"
)
display_records(
    geometry_rows[:15],
    ["component", "upper_name", "lower_name", "reflected_rms_mm"],
)

# This check does not change the already prepared geometry. It makes the
# user's acceptance threshold explicit and raises if any recorded source
# discrepancy exceeds it.
# This example explicitly accepts at most 15 mm RMS point distance.
accepted_geometry_tolerance = 0.015
prepared.check_geometry_tolerance(accepted_geometry_tolerance)
print(f"Accepted geometry tolerance: {1e3 * accepted_geometry_tolerance:.1f} mm")

# Pair records preserve the inferred correspondence, including whether the
# trailing numeric labels happened to agree.
passive_pairs = [
    vars(pair) | {"reflected_rms_mm": 1e3 * pair.reflected_rms}
    for pair in prepared.passive_pairs
]
print(f"Series circuits in source: {tuple(active_series_raw)}")
print(f"Even series circuits retained: {prepared.self_symmetric_active_names}")
print(f"Odd series circuits excluded: {prepared.excluded_odd_active_names}")
print(f"Passive pairs: {len(passive_pairs)}")
print(f"Self-symmetric passive elements: {prepared.self_symmetric_passive_names}")

In [ ]:
# A false matching_numeric_suffix shows why geometry, rather than equal
# suffixes in source labels, must determine the one-to-one assignment.
reordered_cases = [
    row
    for row in passive_pairs
    if "_case_" in row["upper_name"] and not row["matching_numeric_suffix"]
]
print("Case pieces for which matching equal numeric suffixes would be wrong:")
display_records(
    sorted(reordered_cases, key=lambda row: (row["group"], row["upper_name"])),
    ["group", "upper_name", "lower_name", "reflected_rms_mm"],
)

The vessel and centre-column labels also do not encode direct pairs, but their
reflected geometry is distinctive enough for an unambiguous assignment. The
special case visible above is more instructive: for several rectangular coil
cases, `upper_1` pairs with `lower_2` and vice versa. `p5_case` is an especially
clear example.

In [ ]:
# On the left, shift the source to its fitted midplane and reflect each lower
# child so its overlap with the chosen upper child is visible. On the right,
# plot the exactly reflected pair returned in prepared.
raw_passive_by_name = {entry["name"]: entry for entry in passive_raw}
sym_passive_by_name = {entry["name"]: entry for entry in prepared.passive_coils_data}
p5_pairs = [
    pair for pair in prepared.passive_pairs if pair.group == "p5_case"
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for pair_index, pair in enumerate(p5_pairs):
    raw_upper = raw_passive_by_name[pair.upper_name]
    raw_lower = raw_passive_by_name[pair.lower_name]
    upper_r = np.r_[raw_upper["R"], raw_upper["R"][0]]
    upper_z = (
        np.r_[raw_upper["Z"], raw_upper["Z"][0]]
        - prepared.source_z_midplane
    )
    lower_r = np.r_[raw_lower["R"], raw_lower["R"][0]]
    reflected_lower_z = -(
        np.r_[raw_lower["Z"], raw_lower["Z"][0]]
        - prepared.source_z_midplane
    )
    axes[0].plot(
        upper_r,
        upper_z,
        color="tab:blue",
        label="upper pieces" if pair_index == 0 else None,
    )
    axes[0].plot(
        lower_r,
        reflected_lower_z,
        "--",
        color="tab:orange",
        label="reflected lower pieces" if pair_index == 0 else None,
    )
    axes[0].annotate(
        f"U{pair.upper_name.rsplit('_', 1)[-1]}",
        (np.mean(raw_upper["R"]), np.mean(raw_upper["Z"]) - prepared.source_z_midplane),
        color="tab:blue",
        fontsize=8,
    )
    axes[0].annotate(
        f"L{pair.lower_name.rsplit('_', 1)[-1]}",
        (np.mean(raw_lower["R"]), -(np.mean(raw_lower["Z"]) - prepared.source_z_midplane)),
        color="tab:orange",
        fontsize=8,
        xytext=(2, -9),
        textcoords="offset points",
    )

    for name, color in ((pair.upper_name, "tab:blue"), (pair.lower_name, "tab:orange")):
        element = sym_passive_by_name[name]
        r = np.r_[element["R"], element["R"][0]]
        z = np.r_[element["Z"], element["Z"][0]]
        axes[1].plot(r, z, color=color)

axes[0].set_title("Upper and reflected-lower pieces")
axes[1].set_title("Exactly reflected averaged children")
axes[0].legend(fontsize=8)
for axis in axes:
    axis.axhline(0, color="0.3", linewidth=0.8)
    axis.set_xlabel("R [m]")
    axis.set_ylabel("Z [m]")
    axis.set_aspect("equal")
plt.show()

### Pairing outcome for this MAST-U description

- The source contains 12 series-connected active circuits.
- Eleven are electrically even and are retained directly: the solenoid plus
  `px`, `d1`, `d2`, `d3`, `dp`, `d5`, `d6`, `d7`, `p4`, and `p5`.
- `p6` is automatically reported as electrically odd because its lower bundle
  has opposite polarity. Explicit exclusion records it for future prescribed
  forcing; it is never silently inserted into the even machine.
- The 150 passive structures form 75 one-to-one upper/lower pairs.
- The audit reports the largest proposed correction across active circuits,
  passive structures, limiter, and wall before the user accepts a tolerance.

Consequently this machine does not need common-refinement virtual children.
That route is only required when one side has different segmentation, or when
one large element must correspond to several smaller elements.

## Step 4: inspect the averaged geometry, limiter, and wall

For each matched point pair, the upper point and reflected lower point are
averaged. The lower child is then generated by exact reflection of that average.
Material quantities that affect circuit dynamics, such as resistivity and
filament dimensions, are averaged pairwise as well.

The limiter and wall are closed outlines rather than independent elements. Each
outline is split at its two midplane crossings; its upper path and reflected
lower path are resampled by normalized arc length, averaged, and mirrored.

In [ ]:
# Separate the common shift from shape averaging: the first panel shows the
# recentered source and its reflection; the second shows the prepared result.
raw_limiter = np.array([[entry["R"], entry["Z"]] for entry in limiter_raw])
shifted_raw_limiter = raw_limiter.copy()
shifted_raw_limiter[:, 1] += prepared.z_shift
sym_limiter = np.array([[entry["R"], entry["Z"]] for entry in prepared.limiter_data])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), constrained_layout=True)
axes[0].plot(shifted_raw_limiter[:, 0], shifted_raw_limiter[:, 1], label="recentered input")
axes[0].plot(
    shifted_raw_limiter[:, 0],
    -shifted_raw_limiter[:, 1],
    "--",
    label="its reflection",
)
axes[0].set_title("Raw limiter after global shift")
axes[0].legend()

axes[1].plot(sym_limiter[:, 0], sym_limiter[:, 1], label="prepared limiter")
axes[1].plot(sym_limiter[:, 0], -sym_limiter[:, 1], "--", label="its reflection")
axes[1].set_title("Prepared limiter: curves coincide")
axes[1].legend()
for axis in axes:
    axis.axhline(0, color="0.3", linewidth=0.8)
    axis.set_xlabel("R [m]")
    axis.set_ylabel("Z [m]")
    axis.set_aspect("equal")
plt.show()

# The prepared path can start at a different vertex after resampling. Compare
# each point with its nearest reflected point rather than by array index.
reflected_limiter = sym_limiter.copy()
reflected_limiter[:, 1] *= -1
point_distances = np.linalg.norm(
    sym_limiter[:, None, :] - reflected_limiter[None, :, :], axis=-1
)
print(
    "Maximum nearest-point limiter reflection mismatch:",
    f"{np.max(np.min(point_distances, axis=1)):.3e} m",
)

## Step 5: use the series-circuit current basis

There is already one current coordinate per retained active circuit. No
upper/lower current averaging is needed and no independent upper/lower currents
can be reconstructed from this source description.

For the 11 retained circuits, the original-to-even transform is therefore the
identity. P6 remains a separate excluded odd coordinate; retain its source
current separately if a later model explicitly represents odd forcing.

In [ ]:
# Stand-in for the series-circuit currents returned by the symmetric
# example-06a data-loading route.
rng = np.random.default_rng(53320)
series_currents = {
    name: float(value)
    for name, value in zip(
        prepared.original_active_names,
        rng.normal(
            loc=0.0,
            scale=8e3,
            size=len(prepared.original_active_names),
        ),
    )
}

original_active_vector = np.array(
    [series_currents[name] for name in prepared.original_active_names]
)
# Exercise the generic parity-transform API. Here every retained source
# circuit is already internally series-connected and self-symmetric, so the
# even vector is identical to the input and the odd vector is empty.
even_active_vector, odd_active_vector = prepared.split_active_currents(
    original_active_vector
)
round_trip = prepared.combine_active_currents(even_active_vector)
even_projection = round_trip.copy()

print(
    "Series-basis round-trip error:",
    f"{np.max(np.abs(round_trip - original_active_vector)):.3e} A",
)
print(f"Series active coordinates supplied: {len(original_active_vector)}")
print(f"Even active coordinates retained:  {len(even_active_vector)}")
print(f"Odd coordinates inside this prepared machine: {len(odd_active_vector)}")
print(f"Excluded odd source circuits: {odd_series_circuits}")

In [ ]:
active_current_rows = [
    {"circuit": name, "series_current_A": current}
    for name, current in zip(prepared.even_active_names, even_active_vector)
]
display_records(active_current_rows, ["circuit", "series_current_A"])

fig, axis = plt.subplots(figsize=(10, 3.8), constrained_layout=True)
x = np.arange(len(active_current_rows))
axis.bar(
    x,
    np.array([row["series_current_A"] for row in active_current_rows]) / 1e3,
    width=0.65,
    label="retained even series current",
)
axis.set_xticks(
    x,
    [row["circuit"] for row in active_current_rows],
    rotation=45,
)
axis.set_ylabel("series current [kA]")
axis.set_title("The retained active-current basis is already the even basis")
axis.legend()
plt.show()

The preparation changes the bundle geometry and sampled
operators, but it does not change the meaning or units of these 11 series
currents. P6 is kept outside the reduced machine because applying its current
would impose an odd magnetic field.

## Step 6: build the prepared machine

even_active_coils_data contains the 11 retained series circuits after their
internal upper/lower bundle geometry has been averaged. Passive labels remain
unchanged, and their reflection pairing is retained by
passive_reflection_operator for later parity-aware calculations.

In [ ]:
# This is the solver-facing machine. The retained active entries were already
# series circuits; preparation has made their internal bundle geometry exactly
# symmetric. Passive elements retain separate current coordinates.
tokamak = build_machine.tokamak(
    active_coils_data=prepared.even_active_coils_data,
    passive_coils_data=prepared.passive_coils_data,
    limiter_data=prepared.limiter_data,
    wall_data=prepared.wall_data,
)

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.06,
    Rmax=2.0,
    Zmin=-2.2,
    Zmax=2.2,
    nx=65,
    ny=65,
    psi=None,
)

print("Reduced active order:", tokamak.coils_list[: tokamak.n_active_coils])
print(
    f"Machine dimensions: {tokamak.n_active_coils} active + "
    f"{tokamak.n_passive_coils} passive"
)
# The equilibrium grid is symmetric in Z, stored on axis 1. A zero count
# confirms that limiter rasterisation preserved the geometric symmetry.
mask = eq.limiter_handler.mask_inside_limiter
print("Limiter-mask unmatched reflected cells:", np.count_nonzero(mask != mask[:, ::-1]))

## Special cases and limits

This real MAST-U trial establishes what can and cannot be automatic:

1. **Series-connected is not synonymous with even:** P6 is reported as odd and
   requires explicit exclusion. The retained label allows later prescribed
   odd forcing.
2. **Audited before acceptance:** the largest reflected RMS discrepancies are
   reported across active circuits, passives, limiter, and wall. The user sets
   the acceptable geometry tolerance.
3. **Safe automatically here:** internal active-bundle matching, vessel and
   centre-column assignment, reordered coil-case pieces, and the solenoid.
4. **Detected and rejected:** unequal upper/lower counts, incompatible point
   counts, unequal reflected winding magnitudes, or mixed electrical parity.
5. **Requires an override:** ambiguous passive metadata can be resolved with
   explicit `passive_pairs`.
6. **Requires future common refinement:** one large polygon corresponding to
   several polygons on the other side. MAST-U has no such case.
7. **Requires a supplied boundary:** limiter/wall outlines crossing the
   midplane more than twice are intentionally not guessed.

In [ ]:
print("Preparation complete")
print("--------------------")
print(f"source midplane:             {1e3 * prepared.source_z_midplane:+.3f} mm")
print(f"source series circuits:      {len(active_series_raw)}")
print(f"even active circuits used:   {len(prepared.even_active_names)}")
print(f"excluded odd circuits:       {prepared.excluded_odd_active_names}")
print(f"active fingerprint change:   {active_fingerprint_change_percent:.4f}%")
print(f"passive fingerprint change:  {passive_fingerprint_change_percent:.4f}%")
print(f"passive reflected pairs:     {len(prepared.passive_pairs)}")
print(f"self-symmetric passives:     {len(prepared.self_symmetric_passive_names)}")
print(f"limiter mask parity defects: {np.count_nonzero(mask != mask[:, ::-1])}")